# Squiggles Oxford Nanopore Data Pipeline

End-to-end data preparation for AMR prediction: download POD5, basecall+align with Dorado, compile HDF5 feature store.

Requires a Colab GPU runtime (Dorado uses CUDA).

## Modules
1. Environment setup + data download
2. Dorado basecalling and alignment
3. BAM quality audit
4. HDF5 feature store compilation (MAD-normalized signal windows)

Final output: `data/processed/hd5f/amr_features.h5`

In [ ]:
# CELL 1: Mount Google Drive and set project root
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/NanoSquiggle_Pipeline').resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# CELL 2: Create project directories
directories = [
    "bin/dorado/bin",
    "bin/dorado/lib",
    "data/raw",
    "data/processed/alignments",
    "data/processed/hd5f",
    "data/ref",
    "data/data_index",
    "models",
    "scripts"
]

for d in directories:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)
print(f"Created {len(directories)} directories")

In [ ]:
# CELL 3: Download Dorado binary (Colab = Linux x64)
import urllib.request
import tarfile
import shutil

DORADO_VERSION = "0.5.0"
# Colab runs Linux x86_64
DORADO_PLATFORM = "linux-x64"
DORADO_DIR_NAME = f"dorado-{DORADO_VERSION}-{DORADO_PLATFORM}"

dorado_target_bin = PROJECT_ROOT / "bin" / "dorado" / "bin" / "dorado"

if not dorado_target_bin.exists():
    print("Downloading Dorado...")

    dorado_url = f"https://cdn.oxfordnanoportal.com/software/analysis/{DORADO_DIR_NAME}.tar.gz"
    tmp_tar = Path('/content/dorado_temp.tar.gz')
    tmp_dir = Path('/content/dorado_temp_extract')
    tmp_dir.mkdir(exist_ok=True)

    urllib.request.urlretrieve(dorado_url, tmp_tar)
    with tarfile.open(tmp_tar, "r:gz") as tar:
        tar.extractall(path=tmp_dir)

    extracted = tmp_dir / DORADO_DIR_NAME
    for item in (extracted / "bin").iterdir():
        shutil.move(str(item), str(PROJECT_ROOT / "bin" / "dorado" / "bin" / item.name))
    for item in (extracted / "lib").iterdir():
        shutil.move(str(item), str(PROJECT_ROOT / "bin" / "dorado" / "lib" / item.name))

    shutil.rmtree(tmp_dir)
    tmp_tar.unlink()
    dorado_target_bin.chmod(0o755)
    print("Dorado installed")
else:
    print("Dorado already present, skipping")

In [ ]:
# CELL 4: Download + verify POD5 tar.gz for each strain
import os
import tarfile
import shutil
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

class IntegrityChecker:
    """Three-stage integrity: size match, gzip magic bytes, tar structure."""
    @staticmethod
    def validate_magic_bytes(file_path: Path, expected_magic: bytes, offset: int = 0) -> bool:
        try:
            with open(file_path, 'rb') as f:
                f.seek(offset)
                return f.read(len(expected_magic)) == expected_magic
        except Exception as e:
            print(f"Magic byte check failed: {e}")
            return False

    @staticmethod
    def validate_tar_structure(file_path: Path) -> bool:
        try:
            with tarfile.open(file_path, 'r:*') as tar:
                for _ in tar: break
            return True
        except Exception as e:
            print(f"Tar structure check failed: {e}")
            return False

class NanoSquiggleDownloader:
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir

    def download_and_verify(self, url: str, filename: str) -> bool:
        tmp_path = self.output_dir / f"{filename}.tmp"
        file_path = self.output_dir / filename
        print(f"Downloading {filename}...")
        try:
            with requests.get(url, stream=True, timeout=30) as response:
                response.raise_for_status()
                expected_size = int(response.headers.get('Content-Length', 0))
                with open(tmp_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=65536):
                        f.write(chunk)
        except Exception as e:
            print(f"Download failed: {e}")
            if tmp_path.exists(): tmp_path.unlink()
            return False

        # HAC protocol: size -> magic -> tar structure
        local_size = tmp_path.stat().st_size
        if expected_size > 0 and expected_size != local_size:
            print(f"Size mismatch: expected {expected_size}, got {local_size}")
            tmp_path.unlink()
            return False
        if not IntegrityChecker.validate_magic_bytes(tmp_path, b'\x1f\x8b'):
            tmp_path.unlink()
            return False
        if not IntegrityChecker.validate_tar_structure(tmp_path):
            tmp_path.unlink()
            return False

        tmp_path.rename(file_path)
        print(f"{filename} verified OK")
        return True

class DatasetOrchestrator:
    def __init__(self):
        self.project_root = Path(os.getcwd()).resolve()
        self.raw_dir = self.project_root / "data" / "raw"
        self.index_file = self.project_root / "data" / "data_index" / "data_ids.txt"

    def _promote_and_clean(self, target_dir: Path):
        # Flatten pod5_pass if ONT directory structure is present
        pass_dir = target_dir / "pod5_pass"
        fail_dir = target_dir / "pod5_fail"
        if pass_dir.exists() and pass_dir.is_dir():
            for pod5_file in pass_dir.glob("*.pod5"):
                shutil.move(str(pod5_file), str(target_dir / pod5_file.name))
        for folder in [pass_dir, fail_dir]:
            if folder.exists(): shutil.rmtree(folder)

    def extract_and_organize(self, archive_path: Path, target_dir: Path) -> bool:
        # Extract to Colab NVMe (fast), then move to Drive
        print(f"Extracting {target_dir.name} via NVMe buffer...")
        buffer = Path(f"/content/extract_buffer_{target_dir.name}")
        buffer.mkdir(exist_ok=True)
        try:
            with tarfile.open(archive_path, 'r:*') as tar:
                tar.extractall(path=buffer)
            self._promote_and_clean(buffer)
            for item in buffer.iterdir():
                shutil.move(str(item), str(target_dir / item.name))
            archive_path.unlink()
            shutil.rmtree(buffer)
            print(f"{target_dir.name} extracted")
            return True
        except Exception as e:
            print(f"Extraction failed: {e}")
            if buffer.exists(): shutil.rmtree(buffer)
            return False

    def process_strain(self, strain_id: str, url: str) -> bool:
        strain_dir = self.raw_dir / strain_id
        strain_dir.mkdir(parents=True, exist_ok=True)
        sentinel_file = strain_dir / ".download_complete"
        filename = f"{strain_id}.tar.gz"
        archive_path = strain_dir / filename

        if sentinel_file.exists():
            print(f"[SKIP] {strain_id}: already downloaded")
            return True

        downloader = NanoSquiggleDownloader(output_dir=strain_dir)
        if not downloader.download_and_verify(url, filename):
            return False
        if not self.extract_and_organize(archive_path, strain_dir):
            return False
        sentinel_file.touch()
        print(f"[SUCCESS] {strain_id}")
        return True

    def execute_pipeline(self, max_workers: int = 2):
        # max_workers=2 because Drive API penalizes high concurrency
        if not self.index_file.exists():
            print(f"Index not found: {self.index_file}")
            return
        with open(self.index_file, 'r') as f:
            strain_ids = [line.strip() for line in f if line.strip()]
        base_url = "https://data.narodni-repozitar.cz/general/datasets/dj8ys-a4r49/files/"
        tasks = {sid: f"{base_url}{sid}_pod5.tar.gz" for sid in strain_ids}
        print(f"Downloading {len(tasks)} strains...")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.process_strain, sid, url): sid for sid, url in tasks.items()}
            for future in as_completed(futures):
                try:
                    future.result()
                except Exception as e:
                    print(f"Thread error on {strain_id}: {e}")

DatasetOrchestrator().execute_pipeline(max_workers=2)

In [ ]:
# CELL 1: INJECT SYSTEM DEPENDENCIES
# We must install samtools into the Colab Ubuntu instance.
!apt-get update
!apt-get install -y samtools
!samtools --version

In [ ]:
# CELL 2: Dorado basecalling + alignment piped through samtools sort
import subprocess
import os
import sys
import re
import threading
from pathlib import Path

class DoradoRunner:
    def __init__(self):
        self.project_root = Path.cwd().resolve()
        self.raw_dir = self.project_root / "data" / "raw"
        self.processed_dir = self.project_root / "data" / "processed" / "alignments"
        self.dorado_bin = self.project_root / "bin" / "dorado" / "bin" / "dorado"
        self.reference_fasta = self.project_root / "data" / "ref" / "resistance_genes.fasta"
        self.processed_dir.mkdir(parents=True, exist_ok=True)
        try:
            self.dorado_bin.chmod(0o755)
        except Exception as e:
            print(f"Warning: could not chmod dorado: {e}")

    def _get_env(self) -> dict:
        env = os.environ.copy()
        local_lib = str(self.project_root / "bin" / "dorado" / "lib")
        current_ld = env.get("LD_LIBRARY_PATH", "")
        env["LD_LIBRARY_PATH"] = f"{local_lib}:{current_ld}" if current_ld else local_lib
        return env

    def _monitor_telemetry(self, pipe, strain_id: str):
        # Track Dorado throughput from stderr
        throughput_pattern = re.compile(r'([0-9.]+\s+[kM]?(?:samples|bp)/s)')
        for line in iter(pipe.readline, b''):
            decoded_line = line.decode('utf-8', errors='ignore').strip()
            match = throughput_pattern.search(decoded_line)
            if match:
                sys.stderr.write(f"\r{strain_id} speed: {match.group(1)}       ")
                sys.stderr.flush()
            elif any(x in decoded_line.lower() for x in ["error", "warning", "fatal", "info"]):
                if "samples/s" not in decoded_line and "bp/s" not in decoded_line:
                    sys.stderr.write(f"\n{strain_id}: {decoded_line}\n")
        sys.stderr.write("\n")

    def execute_pipeline(self, strain_id: str, pod5_dir: Path) -> bool:
        tmp_bam = self.processed_dir / f"{strain_id}.tmp.bam"
        final_bam = self.processed_dir / f"{strain_id}.bam"
        samtools_log = self.processed_dir / f"{strain_id}_samtools.log"

        if final_bam.exists():
            print(f"[SKIP] {strain_id}: BAM exists")
            return True

        dorado_cmd = [
            str(self.dorado_bin), "basecaller",
            "hac",
            str(pod5_dir),
            "--reference", str(self.reference_fasta),
            "--emit-moves",
            "--emit-sam"
        ]
        samtools_cmd = ["samtools", "sort", "-m", "2G", "-o", str(tmp_bam), "-"]

        print(f"Processing {strain_id}...")
        try:
            p_dorado = subprocess.Popen(
                dorado_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=self._get_env()
            )
            with open(samtools_log, "w") as s_log:
                p_samtools = subprocess.Popen(
                    samtools_cmd, stdin=p_dorado.stdout, stderr=s_log
                )
                p_dorado.stdout.close()
                monitor_thread = threading.Thread(
                    target=self._monitor_telemetry, args=(p_dorado.stderr, strain_id), daemon=True
                )
                monitor_thread.start()
                samtools_exit = p_samtools.wait()
                dorado_exit = p_dorado.wait()
                monitor_thread.join()

            if dorado_exit == 0 and samtools_exit == 0:
                tmp_bam.rename(final_bam)
                print(f"{strain_id}: sorting done, indexing...")
                if samtools_log.exists(): samtools_log.unlink()
                try:
                    subprocess.run(["samtools", "index", str(final_bam)], check=True)
                    return True
                except subprocess.CalledProcessError as e:
                    print(f"Indexing failed: {e}")
                    if final_bam.exists(): final_bam.unlink()
                    return False
            else:
                print(f"{strain_id}: failed (Dorado={dorado_exit}, Samtools={samtools_exit})")
                if tmp_bam.exists(): tmp_bam.unlink()
                return False
        except Exception as e:
            print(f"Error: {e}")
            if tmp_bam.exists(): tmp_bam.unlink()
            return False

    def run(self):
        if not self.raw_dir.exists():
            print(f"Raw dir not found: {self.raw_dir}")
            return
        for strain_folder in self.raw_dir.iterdir():
            if not strain_folder.is_dir(): continue
            sentinel = strain_folder / ".download_complete"
            if not sentinel.exists():
                print(f"Skip {strain_folder.name}: not downloaded")
                continue
            self.execute_pipeline(strain_folder.name, strain_folder)

DoradoRunner().run()

In [ ]:
# CELL 3: HDF5 Feature Store — map BAM alignments to POD5 signal windows
import numpy as np
import h5py
import pod5
import pysam
from pathlib import Path
from collections import defaultdict

!pip install -q pod5 h5py pysam matplotlib seaborn

class FeatureStoreCompiler:
    def __init__(self, target_window=30000, coverage_threshold=0.95, max_background_per_strain=500):
        self.project_root = PROJECT_ROOT
        self.raw_dir = self.project_root / "data" / "raw"
        self.alignments_dir = self.project_root / "data" / "processed" / "alignments"
        self.hdf5_dir = self.project_root / "data" / "processed" / "hd5f"
        self.index_file = self.project_root / "data" / "data_index" / "data_ids.txt"
        self.h5_path = self.hdf5_dir / "amr_features.h5"
        self.error_log_path = self.hdf5_dir / "compilation_errors.log"
        self.target_window = target_window
        self.coverage_threshold = coverage_threshold
        self.max_background_per_strain = max_background_per_strain

        self.pos_genes = [
            "ENA|HEE1644226|HEE1644226.1",
            "ENA|MH733892|MH733892.1",
            "ENA|MZ092836|MZ092836.1"
        ]
        self.neg_genes = [
            "rpoB_1_Klebsiella_pneumoniae_Pasteur",
            "gapA_1_Klebsiella_pneumoniae_Pasteur",
            "mdh_1_Klebsiella_pneumoniae_Pasteur"
        ]

    def _mad_normalize(self, signal):
        median = np.median(signal)
        mad = np.median(np.abs(signal - median))
        if mad == 0:
            mad = 1e-6
        return (signal - median) / mad

    def _extract_spatial_window(self, read, pod5_record):
        # Map alignment coordinates to raw signal indices via Dorado's mv + ts tags
        tags = dict(read.tags)
        if "mv" not in tags or "ts" not in tags:
            return None
        stride = tags["mv"][0]
        moves = np.array(tags["mv"][1:], dtype=np.int32)
        trim_start = tags["ts"]
        cum_moves = np.cumsum(moves)
        moves_to_start = np.searchsorted(cum_moves, read.query_alignment_start, side="left") + 1
        moves_to_end = np.searchsorted(cum_moves, read.query_alignment_end, side="left") + 1
        raw_start = trim_start + (moves_to_start * stride)
        raw_end = trim_start + (moves_to_end * stride)
        signal_midpoint = raw_start + ((raw_end - raw_start) // 2)
        tensor_start = signal_midpoint - (self.target_window // 2)
        tensor_end = signal_midpoint + (self.target_window // 2)
        raw_signal = self._mad_normalize(pod5_record.signal)
        pad_left = max(0, -tensor_start)
        pad_right = max(0, tensor_end - len(raw_signal))
        valid_start = max(0, tensor_start)
        valid_end = min(len(raw_signal), tensor_end)
        sliced_signal = raw_signal[valid_start:valid_end]
        if pad_left > 0 or pad_right > 0:
            sliced_signal = np.pad(sliced_signal, (pad_left, pad_right), mode="constant", constant_values=0)
        return sliced_signal

    def _extract_background_window(self, pod5_record):
        # Centre-crop (or pad) the raw signal to target_window
        raw_signal = self._mad_normalize(pod5_record.signal)
        if len(raw_signal) < self.target_window:
            pad_total = self.target_window - len(raw_signal)
            pad_left = pad_total // 2
            pad_right = pad_total - pad_left
            signal = np.pad(raw_signal, (pad_left, pad_right), mode="constant", constant_values=0)
        else:
            center = len(raw_signal) // 2
            start = center - self.target_window // 2
            signal = raw_signal[start:start + self.target_window]
        return signal

    def _collect_reads_for_strain(self, strain_id):
        bam_path = self.alignments_dir / f"{strain_id}.bam"
        if not bam_path.exists():
            print(f"{strain_id}: no BAM found")
            return []
        samfile = pysam.AlignmentFile(bam_path, "rb")
        reads = []
        for gene, label in ([(g, 1) for g in self.pos_genes] + [(g, 0) for g in self.neg_genes]):
            try:
                ref_len = samfile.get_reference_length(gene)
            except ValueError:
                print(f"{strain_id}: gene '{gene}' not in BAM header, skipping")
                continue
            if ref_len is None or ref_len == 0:
                continue
            gene_count = 0
            for aln in samfile.fetch(reference=gene):
                if not aln.is_mapped: continue
                rlen = aln.reference_length
                if rlen is None or rlen == 0: continue
                if rlen / ref_len >= self.coverage_threshold:
                    reads.append({"read_id": aln.query_name, "label": label, "strain": strain_id, "gene": gene, "sam_read": aln})
                    gene_count += 1
            print(f"{strain_id}: {gene_count} reads for target {gene.split('|')[-1] if '|' in gene else gene}")
        bg = 0
        for aln in samfile.fetch(until_eof=True):
            if aln.is_unmapped:
                reads.append({"read_id": aln.query_name, "label": 0, "strain": strain_id, "gene": "background", "sam_read": aln, "type": "background"})
                bg += 1
                if bg >= self.max_background_per_strain:
                    break
        samfile.close()
        return reads

    def compile(self):
        print("Compiling feature store...")
        if not self.index_file.exists():
            print(f"Index not found: {self.index_file}")
            return
        with open(self.index_file) as f:
            strain_ids = [l.strip() for l in f if l.strip()]
        all_reads = []
        for sid in strain_ids:
            reads = self._collect_reads_for_strain(sid)
            print(f"  {sid}: {len(reads)} reads")
            all_reads.extend(reads)
        N = len(all_reads)
        print(f"Total: {N} reads")
        if N == 0:
            return
        self.hdf5_dir.mkdir(parents=True, exist_ok=True)
        dt_str = h5py.string_dtype(encoding="utf-8")
        by_strain = defaultdict(list)
        for r in all_reads:
            by_strain[r["strain"]].append(r)
        errors = []
        with h5py.File(self.h5_path, "w") as h5f:
            X = h5f.create_dataset("X", shape=(N, self.target_window), maxshape=(None, self.target_window), dtype=np.float32)
            Y = h5f.create_dataset("Y", shape=(N,), maxshape=(None,), dtype=np.int8)
            M_rid = h5f.create_dataset("read_id", shape=(N,), maxshape=(None,), dtype=dt_str)
            M_sid = h5f.create_dataset("strain_id", shape=(N,), maxshape=(None,), dtype=dt_str)
            M_gen = h5f.create_dataset("gene_target", shape=(N,), maxshape=(None,), dtype=dt_str)
            idx = 0
            success_by_strain = defaultdict(int)
            for strain_id, strain_reads in by_strain.items():
                pod5_dir = self.raw_dir / strain_id
                pod5_files = sorted(pod5_dir.glob("*.pod5"))
                if not pod5_files:
                    print(f"  {strain_id}: no POD5 files")
                    for r in strain_reads:
                        errors.append((r["read_id"], strain_id, "No POD5 files"))
                    continue
                pending = {r["read_id"]: r for r in strain_reads}
                for pod5_file in pod5_files:
                    if not pending:
                        break
                    remaining = list(pending.keys())
                    with pod5.Reader(pod5_file) as reader:
                        for record in reader.reads(remaining, missing_ok=True):
                            rid = str(record.read_id)
                            meta = pending.pop(rid, None)
                            if meta is None:
                                continue
                            try:
                                if meta.get("type") == "background":
                                    signal = self._extract_background_window(record)
                                else:
                                    signal = self._extract_spatial_window(meta["sam_read"], record)
                                if signal is None:
                                    raise ValueError("Missing mv/ts tags")
                                X[idx] = signal
                                Y[idx] = meta["label"]
                                M_rid[idx] = rid
                                M_sid[idx] = strain_id
                                M_gen[idx] = meta["gene"]
                                idx += 1
                                success_by_strain[strain_id] += 1
                                if idx % 100 == 0:
                                    print(f"    -> {idx} / {N}")
                            except Exception as e:
                                errors.append((rid, strain_id, str(e)))
                for rid, meta in pending.items():
                    errors.append((rid, strain_id, "Read not found in POD5"))
            actual = idx
            if actual < N:
                X.resize((actual, self.target_window))
                Y.resize((actual,))
                M_rid.resize((actual,))
                M_sid.resize((actual,))
                M_gen.resize((actual,))
        if errors:
            with open(self.error_log_path, "w") as f:
                for rid, sid, err in errors:
                    f.write(f"{rid},{sid},{err}\n")
            print(f"{len(errors)} errors logged to {self.error_log_path}")
        print(f"Feature store: {self.h5_path}")
        print(f"Tensors written: {actual}")
        if actual < N:
            print(f"{N - actual}/{N} reads failed ({((N - actual) / N) * 100:.1f}%)")
        for sid in sorted(success_by_strain):
            n_req = len(by_strain[sid])
            n_written = success_by_strain[sid]
            if n_written < n_req:
                print(f"  {sid}: {n_written}/{n_req}")
            else:
                print(f"  {sid}: {n_written}")

compiler = FeatureStoreCompiler()
compiler.compile()

In [ ]:
# CELL 4: Inspect feature store — class balance, signal stats, per-class plots
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

H5_PATH = PROJECT_ROOT / "data" / "processed" / "hd5f" / "amr_features.h5"

f = h5py.File(H5_PATH, "r")
x = f["X"][:]
y = f["Y"][:]
strain = f["strain_id"][:]
gene = f["gene_target"][:]
N = len(y)

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total samples: {N}")
print(f"X shape: {x.shape}, dtype: {x.dtype}")
print(f"Signal range: [{x.min():.4f}, {x.max():.4f}]")
print(f"Signal mean: {x.mean():.4f}, std: {x.std():.4f}\n")

print("CLASS DISTRIBUTION")
pos = int(np.sum(y == 1))
neg = int(np.sum(y == 0))
print(f"  AMR (label=1):     {pos:5d} ({pos/N*100:.1f}%)")
print(f"  Non-AMR (label=0): {neg:5d} ({neg/N*100:.1f}%)\n")

print("STRAIN DISTRIBUTION")
for s in np.unique(strain):
    print(f"  {s.decode()}: {int(np.sum(strain == s))}")
print()

print("GENE DISTRIBUTION")
for g in np.unique(gene):
    print(f"  {g.decode()}: {int(np.sum(gene == g))}")
print()

extreme = int(np.sum(np.any(np.abs(x) > 50, axis=1)))
print(f"Extreme windows (>50 MAD): {extreme}")
all_zero = int(np.sum(np.all(np.abs(x) < 1e-6, axis=1)))
print(f"All-zero windows: {all_zero}\n")

# Per-class signal overlay plot
sns.set_style("whitegrid")
fig, axes = plt.subplots(3, 1, figsize=(14, 8))
categories = [
    ("AMR Positive", y == 1, "#e74c3c"),
    ("MLST Hard Negative", (y == 0) & (gene != b"background"), "#3498db"),
    ("Background", gene == b"background", "#95a5a6"),
]
for ax, (title, mask, color) in zip(axes, categories):
    indices = np.where(mask)[0]
    if len(indices) == 0:
        ax.text(0.5, 0.5, "No samples", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        continue
    samples = np.random.choice(indices, min(5, len(indices)), replace=False)
    offset = 0
    for idx in samples:
        ax.plot(x[idx] + offset, color=color, alpha=0.8, linewidth=0.5)
        offset += 4
    ax.set_ylabel("Signal (MAD)")
    ax.set_title(f"{title} ({len(indices)} reads)")
    ax.set_yticks([])
axes[-1].set_xlabel("Sample index")
plt.tight_layout()
plot_path = PROJECT_ROOT / "data" / "processed" / "hd5f" / "signal_quality_check.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved to {plot_path}")
f.close()